### Analysis code that removes question bias and generates detailed legal reports

In [1]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

# --- CONFIGURATION ---
TRUTH_FILE = "subset_validation.csv"
OUTPUT_FILE = "Port_Taiwan_US_pivot.csv" 
QUESTIONS_JSON = "organized_constitutional_questions.json"
DEPENDENCY_PLAN = "dependency_plan_levels_organized_constitutional_questions.json"
RAW_OUTPUT_FILE = "Port_Taiwan_US_dummy.csv" 
ID_COLS = ["country", "year"]
RESULTS_DIR = "Validation_Results_Final"

# Ensure output directory exists
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

# ==============================================================================
# 1. HELPER FUNCTIONS
# ==============================================================================

def clean_val(v):
    s = str(v).strip()
    if s.endswith(".0"): s = s[:-2]
    if s.lower() in ["nan", "none", "", "missing", "-1"]: return "-1"
    return s.lower()

def to_relaxed(val):
    s = clean_val(val)
    try:
        n = int(float(s))
        if n in [96, 97, 98, 99]: return "0"
        return str(n)
    except:
        return s

def calc_full_metrics(y_true, y_pred, label):
    if len(y_true) == 0: 
        return {k: 0.0 for k in [f"{label}Acc", f"{label}F1", f"{label}Prec", f"{label}Rec"]}
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return {f"{label}Acc": round(acc, 4), f"{label}Prec": round(p, 4), f"{label}Rec": round(r, 4), f"{label}F1": round(f1, 4)}

# ==============================================================================
# 2. HYBRID UNBIASED METRICS (CONSOLE OUTPUT)
# ==============================================================================

def run_hybrid_unbiased_analysis():
    truth_df = pd.read_csv(TRUTH_FILE)
    output_df = pd.read_csv(OUTPUT_FILE)
    
    for df in [truth_df, output_df]:
        if "country" in df.columns:
            df["country"] = df["country"].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})

    with open(QUESTIONS_JSON, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    with open(DEPENDENCY_PLAN, "r", encoding="utf-8") as f:
        plan = json.load(f)

    merged_df = pd.merge(truth_df, output_df, on=ID_COLS, suffixes=("_truth", "_pred"))
    var_to_parent, parent_type, multi_select_parents = {}, {}, set()
    root_vars = set(plan[0])

    for chunk in q_data.get("chunks", {}).values():
        for q in chunk.get("questions", []):
            p_id = q.get("code") or q.get("id")
            var_to_parent[p_id] = p_id
            parent_type[p_id] = "Root" if p_id in root_vars else "Cond"
            if q.get("multi_select") is True: multi_select_parents.add(p_id)
            for opt in q.get("options", []):
                if opt.get("code"): var_to_parent[opt.get("code")] = p_id

    valid_cols = [c for c in truth_df.columns if c in output_df.columns and c not in ID_COLS]
    parent_groups = {}
    for var in valid_cols:
        parent = var_to_parent.get(var, var)
        if parent not in parent_groups: parent_groups[parent] = []
        parent_groups[parent].append(var)

    v = {k: {"t": [], "p": []} for k in ["GS", "GR", "RS", "RR", "CAS", "CAR", "CSS", "CSR"]}
    for _, row in merged_df.iterrows():
        for parent, children in parent_groups.items():
            is_multi = parent in multi_select_parents
            if is_multi:
                ts, ps = [clean_val(row[c+"_truth"]) for c in children], [clean_val(row[c+"_pred"]) for c in children]
                tr, pr = [to_relaxed(row[c+"_truth"]) for c in children], [to_relaxed(row[c+"_pred"]) for c in children]
            else:
                ts, ps = ["|".join([clean_val(row[c+"_truth"]) for c in children])], ["|".join([clean_val(row[c+"_pred"]) for c in children])]
                tr, pr = ["|".join([to_relaxed(row[c+"_truth"]) for c in children])], ["|".join([to_relaxed(row[c+"_pred"]) for c in children])]

            h_skip = clean_val(row[children[0] + "_truth"]) == "99"
            p_type = parent_type.get(parent)
            for t_s, p_s, t_r, p_r in zip(ts, ps, tr, pr):
                v["GS"]["t"].append(t_s); v["GS"]["p"].append(p_s); v["GR"]["t"].append(t_r); v["GR"]["p"].append(p_r)
                if p_type == "Root":
                    v["RS"]["t"].append(t_s); v["RS"]["p"].append(p_s); v["RR"]["t"].append(t_r); v["RR"]["p"].append(p_r)
                elif h_skip:
                    v["CSS"]["t"].append(t_s); v["CSS"]["p"].append(p_s); v["CSR"]["t"].append(t_r); v["CSR"]["p"].append(p_r)
                else:
                    v["CAS"]["t"].append(t_s); v["CAS"]["p"].append(p_s); v["CAR"]["t"].append(t_r); v["CAR"]["p"].append(p_r)

    print(f"\n{'='*70}\n{'HYBRID UNBIASED METRICS SUMMARY':^70}\n{'='*70}")
    def p_m(title, data, lab):
        m = calc_full_metrics(data["t"], data["p"], lab)
        print(f"\n--- {title} ---\nAcc: {m[lab+'Acc']:.2%} | F1: {m[lab+'F1']:.4f} | Prec: {m[lab+'Prec']:.4f}")
    
    p_m("GLOBAL STRICT", v["GS"], "S")
    p_m("GLOBAL RELAXED", v["GR"], "R")
    
    print(f"\n{'-'*40}\n HIERARCHICAL (Strict)\n{'-'*40}")
    p_m("ROOT", v["RS"], "S")
    p_m("COND: ANSWER", v["CAS"], "S")
    p_m("COND: SKIP", v["CSS"], "S")
    
    print(f"\n{'-'*40}\n HIERARCHICAL (Relaxed)\n{'-'*40}")
    p_m("ROOT", v["RR"], "R")
    p_m("COND: ANSWER", v["CAR"], "R")
    p_m("COND: SKIP", v["CSR"], "R")

# ==============================================================================
# 3. REPORT GENERATION (EXCEL)
# ==============================================================================

def load_ai_reasoning(csv_path):
    reasoning_map = {}
    try:
        df = pd.read_csv(csv_path)
        df['country'] = df['country'].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})
        for _, row in df.iterrows():
            key = (str(row["country"]), str(row["year"]), str(row["variable_name"]))
            reasoning_map[key] = str(row["explanation"]).replace("\n", " ").strip()
    except Exception as e: print(f"Warning: Reasoning load failed: {e}")
    return reasoning_map

def load_question_metadata(json_path):
    metadata = {}
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for chunk in data.get("chunks", {}).values():
            for q in chunk.get("questions", []):
                code = q.get("code") or q.get("id")
                if not code: continue
                cond = q["conditional"].get("raw", "None") if isinstance(q.get("conditional"), dict) else "None"
                opts = {str(o["number"]): o["text"] for o in q.get("options", [])}
                metadata[code] = {"id": q.get("id", ""), "text": q.get("question", ""), "condition": cond, "options": opts, "instructions": q.get("instructions", "")}
    except Exception: pass
    return metadata

def create_validation_detailed_report():
    """Generates Validation_Detailed_Report.xlsx with Codebook and Match status."""
    truth_df = pd.read_csv(TRUTH_FILE)
    output_df = pd.read_csv(OUTPUT_FILE)
    for df in [truth_df, output_df]: 
        df['country'] = df['country'].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})
    
    metadata = load_question_metadata(QUESTIONS_JSON)
    reasoning_map = load_ai_reasoning(RAW_OUTPUT_FILE)
    merged_df = pd.merge(truth_df, output_df, on=ID_COLS, suffixes=("_truth", "_pred"))
    
    with open(DEPENDENCY_PLAN, "r", encoding="utf-8") as f: plan = json.load(f)
    root_vars = set(plan[0])
    valid_cols = sorted(list(set(truth_df.columns).intersection(output_df.columns) - set(ID_COLS)))
    
    wb = Workbook()
    ws_data = wb.active
    ws_data.title = "Data"
    
    # Codebook Sheet
    ws_cb = wb.create_sheet(title="Codebook_Options")
    ws_cb.append(["Variable_Code", "Question_Text", "Conditional_Raw", "Option_Number", "Option_Text", "Instructions"])
    fill_h = PatternFill(start_color="DDDDDD", end_color="DDDDDD", fill_type="solid")
    for cell in ws_cb[1]: cell.font = Font(bold=True); cell.fill = fill_h

    for code, q in sorted(metadata.items()):
        base = code.rsplit("_", 1)[0] if "_" in code and code.rsplit("_", 1)[1].isdigit() else code
        if q['options'] and base == code:
            for num, txt in q['options'].items(): ws_cb.append([code, q['text'], q['condition'], num, txt, q['instructions']])
        elif not q['options'] and base == code: ws_cb.append([code, q['text'], q['condition'], "Open-Ended", "See Instructions", q['instructions']])

    # Data Comparison Sheet
    ws_data.append(["Country", "Year", "Variable", "Type", "ID", "Question", "Condition", "Human Code", "Human Text", "AI Code", "AI Text", "AI Reasoning", "Strict Match", "Relaxed Match"])
    for cell in ws_data[1]: cell.font = Font(bold=True); cell.fill = fill_h

    fill_g, fill_y, fill_r = [PatternFill(start_color=c, end_color=c, fill_type="solid") for c in ["C6EFCE", "FFEB9C", "FFC7CE"]]

    for _, row in merged_df.iterrows():
        c, y = str(row["country"]), str(row["year"])
        for col in valid_cols:
            vt, vp = row[col + "_truth"], row[col + "_pred"]
            strict = (clean_val(vt) == clean_val(vp))
            relaxed = (to_relaxed(vt) == to_relaxed(vp))
            base_col = col.rsplit("_", 1)[0] if "_" in col and col.rsplit("_", 1)[1].isdigit() else col
            q_info = metadata.get(base_col, {"id": "N/A", "text": col, "condition": "Unknown", "options": {}})
            
            h_txt = f"{vt} ({q_info['options'].get(clean_val(vt), 'Other')})"
            a_txt = f"{vp} ({q_info['options'].get(clean_val(vp), 'Other')})"
            reason = reasoning_map.get((c, y, col), "")

            ws_data.append([c, y, col, "Root" if base_col in root_vars else "Cond", q_info['id'], q_info['text'], q_info['condition'], vt, h_txt, vp, a_txt, reason, "YES" if strict else "NO", "YES" if relaxed else "NO"])
            color = fill_g if strict else (fill_y if relaxed else fill_r)
            for idx in [9, 11, 13, 14]: ws_data.cell(row=ws_data.max_row, column=idx).fill = color

    report_path = os.path.join(RESULTS_DIR, "Validation_Detailed_Report.xlsx")
    wb.save(report_path)
    print(f"✅ Validation_Detailed_Report.xlsx saved to {RESULTS_DIR}")

def generate_detailed_comparison_report(json_path, truth_csv, ai_pivot_csv):
    """Generates Detailed_Comparison_Report.xlsx with strict conditional formatting."""
    output_excel = os.path.join(RESULTS_DIR, "Detailed_Comparison_Report.xlsx")
    
    with open(json_path, 'r', encoding='utf-8') as f: meta = json.load(f)
    question_map, option_detail_map = {}, {}

    for chunk in meta.get("chunks", {}).values():
        for q in chunk.get("questions", []):
            q_text = q.get("question", "No text found")
            instr, cond_expr = q.get("instructions") or "", q.get("conditional", {}).get("condition_expression") or ""
            q_type = "Conditional" if q.get("conditional", {}).get("depends_on") else "Root"
            base_code = q.get("code") or q.get("id")
            question_map[base_code] = {"text": q_text, "instr": instr, "cond": cond_expr, "type": q_type}
            for opt in q.get("options", []):
                opt_code = opt.get("code")
                if opt_code:
                    option_detail_map[opt_code] = {"q_text": q_text, "opt_label": opt.get("text", "N/A"), "instr": instr, "cond": cond_expr, "type": q_type}

    df_human, df_ai = pd.read_csv(truth_csv), pd.read_csv(ai_pivot_csv)
    for df in [df_human, df_ai]:
        df['country'] = df['country'].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})
        df['id'] = df['country'] + "_" + df['year'].astype(str)

    id_cols = ['country', 'year', 'id']
    shared_variables = sorted(list(set(df_human.columns).intersection(set(df_ai.columns)) - set(id_cols)))
    constitutions, rows = sorted(df_human['id'].unique()), []
    
    for var in shared_variables:
        if var in option_detail_map:
            d = option_detail_map[var]
            q_text, opt_label, instr, cond, q_type = d["q_text"], d["opt_label"], d["instr"], d["cond"], d["type"]
        else:
            d = question_map.get(var, {})
            q_text, opt_label, instr, cond, q_type = d.get("text", "N/A"), "Standard", d.get("instr", ""), d.get("cond", ""), d.get("type", "N/A")
        
        row = {"Variable": var, "Type": q_type, "Question": q_text, "Option": opt_label, "Instructions": instr, "Condition": cond}
        for const_id in constitutions:
            h_val = df_human[df_human['id'] == const_id][var].values[0] if not df_human[df_human['id'] == const_id].empty else None
            a_val = df_ai[df_ai['id'] == const_id][var].values[0] if not df_ai[df_ai['id'] == const_id].empty else None
            row[f"{const_id}_H"], row[f"{const_id}_A"] = h_val, a_val
            h_c, a_c, h_r, a_r = clean_val(h_val), clean_val(a_val), to_relaxed(h_val), to_relaxed(a_val)
            row[f"{const_id}_Match"] = "Exact" if h_c == a_c else ("Relaxed" if h_r == a_r else "None")
        rows.append(row)

    final_df = pd.DataFrame(rows)
    writer = pd.ExcelWriter(output_excel, engine='xlsxwriter')
    final_df.to_excel(writer, index=False, sheet_name='Comparison')
    workbook, worksheet = writer.book, writer.sheets['Comparison']
    
    green = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})
    yellow = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500'})
    red = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})
    
    for col_num, col_name in enumerate(final_df.columns):
        if col_name.endswith('_Match'):
            worksheet.conditional_format(1, col_num, len(final_df), col_num, {'type': 'cell', 'criteria': 'equal to', 'value': '"Exact"', 'format': green})
            worksheet.conditional_format(1, col_num, len(final_df), col_num, {'type': 'cell', 'criteria': 'equal to', 'value': '"Relaxed"', 'format': yellow})
            worksheet.conditional_format(1, col_num, len(final_df), col_num, {'type': 'cell', 'criteria': 'equal to', 'value': '"None"', 'format': red})

    writer.close()
    print(f"✅ Detailed_Comparison_Report.xlsx saved to {RESULTS_DIR}")

### Main execution

In [2]:
if __name__ == "__main__":
    # 1. Console accuracy summary
    run_hybrid_unbiased_analysis()
    
    # 2. Excel Reports
    create_validation_detailed_report()
    generate_detailed_comparison_report(QUESTIONS_JSON, TRUTH_FILE, OUTPUT_FILE)


                   HYBRID UNBIASED METRICS SUMMARY                    

--- GLOBAL STRICT ---
Acc: 54.81% | F1: 0.5356 | Prec: 0.5548

--- GLOBAL RELAXED ---
Acc: 63.46% | F1: 0.6139 | Prec: 0.6410

----------------------------------------
 HIERARCHICAL (Strict)
----------------------------------------

--- ROOT ---
Acc: 74.29% | F1: 0.7378 | Prec: 0.7664

--- COND: ANSWER ---
Acc: 50.37% | F1: 0.5145 | Prec: 0.5587

--- COND: SKIP ---
Acc: 35.90% | F1: 0.4463 | Prec: 0.5897

----------------------------------------
 HIERARCHICAL (Relaxed)
----------------------------------------

--- ROOT ---
Acc: 75.92% | F1: 0.7578 | Prec: 0.7724

--- COND: ANSWER ---
Acc: 55.01% | F1: 0.5228 | Prec: 0.5451

--- COND: SKIP ---
Acc: 66.03% | F1: 0.7163 | Prec: 0.9872
✅ Validation_Detailed_Report.xlsx saved to Validation_Results_Final
✅ Detailed_Comparison_Report.xlsx saved to Validation_Results_Final


### Level-Sorted Version

In [3]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

# --- CONFIGURATION ---
TRUTH_FILE = "subset_validation.csv"
OUTPUT_FILE = "Port_Taiwan_US_pivot.csv" 
RAW_OUTPUT_FILE = "Port_Taiwan_US_dummy.csv" 
QUESTIONS_JSON = "organized_constitutional_questions.json"
DEPENDENCY_PLAN = "dependency_plan_levels_organized_constitutional_questions.json"
ID_COLS = ["country", "year"]
RESULTS_DIR = "Validation_Results_Final"

if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

# ==============================================================================
# 1. HELPER FUNCTIONS
# ==============================================================================

def clean_val(v):
    s = str(v).strip()
    if s.endswith(".0"): s = s[:-2]
    if s.lower() in ["nan", "none", "", "missing", "-1"]: return "-1"
    return s.lower()

def to_relaxed(val):
    s = clean_val(val)
    try:
        n = int(float(s))
        if n in [96, 97, 98, 99]: return "0"
        return str(n)
    except:
        return s

def calc_full_metrics(y_true, y_pred, label):
    if len(y_true) == 0: 
        return {k: 0.0 for k in [f"{label}Acc", f"{label}F1", f"{label}Prec", f"{label}Rec"]}
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return {f"{label}Acc": round(acc, 4), f"{label}Prec": round(p, 4), f"{label}Rec": round(r, 4), f"{label}F1": round(f1, 4)}

# ==============================================================================
# 2. HYBRID UNBIASED METRICS (CONSOLE OUTPUT)
# ==============================================================================

def run_hybrid_unbiased_analysis():
    truth_df = pd.read_csv(TRUTH_FILE)
    output_df = pd.read_csv(OUTPUT_FILE)
    
    for df in [truth_df, output_df]:
        if "country" in df.columns:
            df["country"] = df["country"].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})

    with open(QUESTIONS_JSON, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    with open(DEPENDENCY_PLAN, "r", encoding="utf-8") as f:
        plan = json.load(f)

    merged_df = pd.merge(truth_df, output_df, on=ID_COLS, suffixes=("_truth", "_pred"))
    var_to_parent, parent_type, multi_select_parents = {}, {}, set()
    root_vars = set(plan[0])

    for chunk in q_data.get("chunks", {}).values():
        for q in chunk.get("questions", []):
            p_id = q.get("code") or q.get("id")
            var_to_parent[p_id] = p_id
            parent_type[p_id] = "Root" if p_id in root_vars else "Cond"
            if q.get("multi_select") is True: multi_select_parents.add(p_id)
            for opt in q.get("options", []):
                if opt.get("code"): var_to_parent[opt.get("code")] = p_id

    valid_cols = [c for c in truth_df.columns if c in output_df.columns and c not in ID_COLS]
    parent_groups = {}
    for var in valid_cols:
        parent = var_to_parent.get(var, var)
        if parent not in parent_groups: parent_groups[parent] = []
        parent_groups[parent].append(var)

    v = {k: {"t": [], "p": []} for k in ["GS", "GR", "RS", "RR", "CAS", "CAR", "CSS", "CSR"]}
    for _, row in merged_df.iterrows():
        for parent, children in parent_groups.items():
            is_multi = parent in multi_select_parents
            if is_multi:
                ts, ps = [clean_val(row[c+"_truth"]) for c in children], [clean_val(row[c+"_pred"]) for c in children]
                tr, pr = [to_relaxed(row[c+"_truth"]) for c in children], [to_relaxed(row[c+"_pred"]) for c in children]
            else:
                ts, ps = ["|".join([clean_val(row[c+"_truth"]) for c in children])], ["|".join([clean_val(row[c+"_pred"]) for c in children])]
                tr, pr = ["|".join([to_relaxed(row[c+"_truth"]) for c in children])], ["|".join([to_relaxed(row[c+"_pred"]) for c in children])]

            h_skip = clean_val(row[children[0] + "_truth"]) == "99"
            p_type = parent_type.get(parent)
            for t_s, p_s, t_r, p_r in zip(ts, ps, tr, pr):
                v["GS"]["t"].append(t_s); v["GS"]["p"].append(p_s); v["GR"]["t"].append(t_r); v["GR"]["p"].append(p_r)
                if p_type == "Root":
                    v["RS"]["t"].append(t_s); v["RS"]["p"].append(p_s); v["RR"]["t"].append(t_r); v["RR"]["p"].append(p_r)
                elif h_skip:
                    v["CSS"]["t"].append(t_s); v["CSS"]["p"].append(p_s); v["CSR"]["t"].append(t_r); v["CSR"]["p"].append(p_r)
                else:
                    v["CAS"]["t"].append(t_s); v["CAS"]["p"].append(p_s); v["CAR"]["t"].append(t_r); v["CAR"]["p"].append(p_r)

    print(f"\n{'='*70}\n{'HYBRID UNBIASED METRICS SUMMARY':^70}\n{'='*70}")
    def p_m(title, data, lab):
        m = calc_full_metrics(data["t"], data["p"], lab)
        print(f"\n--- {title} ---\nAcc: {m[lab+'Acc']:.2%} | F1: {m[lab+'F1']:.4f} | Prec: {m[lab+'Prec']:.4f}")
    
    p_m("GLOBAL STRICT", v["GS"], "S")
    p_m("GLOBAL RELAXED", v["GR"], "R")
    
    print(f"\n{'-'*40}\n HIERARCHICAL (Strict)\n{'-'*40}")
    p_m("ROOT", v["RS"], "S")
    p_m("COND: ANSWER", v["CAS"], "S")
    p_m("COND: SKIP", v["CSS"], "S")
    
    print(f"\n{'-'*40}\n HIERARCHICAL (Relaxed)\n{'-'*40}")
    p_m("ROOT", v["RR"], "R")
    p_m("COND: ANSWER", v["CAR"], "R")
    p_m("COND: SKIP", v["CSR"], "R")

# ==============================================================================
# 3. CONSOLIDATED EXCEL GENERATION (LEVEL-SORTED & GROUPED)
# ==============================================================================

def create_consolidated_report():
    truth_df = pd.read_csv(TRUTH_FILE)
    output_df = pd.read_csv(OUTPUT_FILE)
    for df in [truth_df, output_df]: 
        df['country'] = df['country'].astype(str).str.strip().str.title().replace({"United States Of America": "United States"})
        df['id'] = df['country'] + "_" + df['year'].astype(str)

    with open(QUESTIONS_JSON, "r", encoding="utf-8") as f: q_data = json.load(f)
    with open(DEPENDENCY_PLAN, "r", encoding="utf-8") as f: plan_levels = json.load(f)
    
    # 1. Map Variables to Parent Questions for Level Sorting
    var_to_parent = {}
    q_metadata = {}
    for chunk in q_data.get("chunks", {}).values():
        for q in chunk.get("questions", []):
            p_id = q.get("code") or q.get("id")
            var_to_parent[p_id] = p_id
            q_metadata[p_id] = {"text": q.get("question", ""), "condition": q.get("conditional", {}).get("raw", "None"), "options": {str(o["number"]): o["text"] for o in q.get("options", [])}, "instr": q.get("instructions", "")}
            for opt in q.get("options", []):
                if opt.get("code"): var_to_parent[opt.get("code")] = p_id

    level_map = {code: idx for idx, codes in enumerate(plan_levels) for code in codes}
    constitutions = sorted(truth_df['id'].unique())
    valid_cols = sorted(list(set(truth_df.columns).intersection(output_df.columns) - set(ID_COLS) - {"id"}))
    
    wb = Workbook()
    fill_h = PatternFill(start_color="DDDDDD", end_color="DDDDDD", fill_type="solid")
    fill_g, fill_y, fill_r = [PatternFill(start_color=c, end_color=c, fill_type="solid") for c in ["C6EFCE", "FFEB9C", "FFC7CE"]]

    # --- SHEET 1: LEVEL-SORTED COMPARISON (Create this first) ---
    ws_comp = wb.active
    ws_comp.title = "Level_Sorted_Comparison"
    headers = ["Level", "Parent_Code", "Variable", "Type", "Question", "Condition"]
    for cid in constitutions: headers.extend([f"{cid}_H", f"{cid}_A", f"{cid}_Match"])
    ws_comp.append(headers)
    for cell in ws_comp[1]: cell.font = Font(bold=True); cell.fill = fill_h
    
    # --- SHEET 2: CODEBOOK (Create this second) ---
    ws_cb = wb.create_sheet(title="Codebook_Options")
    ws_cb.append(["Variable_Code", "Question_Text", "Conditional_Raw", "Option_Number", "Option_Text", "Instructions"])
    for cell in ws_cb[1]: cell.font = Font(bold=True); cell.fill = fill_h
    for code, q in sorted(q_metadata.items()):
        if q['options']:
            for num, txt in q['options'].items(): ws_cb.append([code, q['text'], q['condition'], num, txt, q['instr']])
        else: ws_cb.append([code, q['text'], q['condition'], "Open-Ended", "N/A", q['instr']])

    comp_rows = []
    for var in valid_cols:
        parent = var_to_parent.get(var, var)
        lvl = level_map.get(parent, 99)
        q_info = q_metadata.get(parent, {"text": var, "condition": "Unknown"})
        row = [lvl, parent, var, "Root" if lvl == 0 else "Cond", q_info['text'], q_info['condition']]
        for cid in constitutions:
            h_val = truth_df[truth_df['id'] == cid][var].values[0] if not truth_df[truth_df['id'] == cid].empty else None
            a_val = output_df[output_df['id'] == cid][var].values[0] if not output_df[output_df['id'] == cid].empty else None
            status = "Exact" if clean_val(h_val) == clean_val(a_val) else ("Relaxed" if to_relaxed(h_val) == to_relaxed(a_val) else "None")
            row.extend([h_val, a_val, status])
        comp_rows.append(row)
    
    # Sort by Level, then Parent, then Variable name
    for row in sorted(comp_rows, key=lambda x: (x[0], x[1], x[2])):
        ws_comp.append(row)
        for i, status in enumerate(row[8::3]): # Match columns
            cell = ws_comp.cell(row=ws_comp.max_row, column=9 + (i*3))
            cell.fill = fill_g if status == "Exact" else (fill_y if status == "Relaxed" else fill_r)

    report_path = os.path.join(RESULTS_DIR, "Constitutional_Validation_Level_Sorted.xlsx")
    wb.save(report_path)
    print(f"\n✅ All results consolidated into: {report_path}")

if __name__ == "__main__":
    run_hybrid_unbiased_analysis()
    create_consolidated_report()


                   HYBRID UNBIASED METRICS SUMMARY                    

--- GLOBAL STRICT ---
Acc: 54.81% | F1: 0.5356 | Prec: 0.5548

--- GLOBAL RELAXED ---
Acc: 63.46% | F1: 0.6139 | Prec: 0.6410

----------------------------------------
 HIERARCHICAL (Strict)
----------------------------------------

--- ROOT ---
Acc: 74.29% | F1: 0.7378 | Prec: 0.7664

--- COND: ANSWER ---
Acc: 50.37% | F1: 0.5145 | Prec: 0.5587

--- COND: SKIP ---
Acc: 35.90% | F1: 0.4463 | Prec: 0.5897

----------------------------------------
 HIERARCHICAL (Relaxed)
----------------------------------------

--- ROOT ---
Acc: 75.92% | F1: 0.7578 | Prec: 0.7724

--- COND: ANSWER ---
Acc: 55.01% | F1: 0.5228 | Prec: 0.5451

--- COND: SKIP ---
Acc: 66.03% | F1: 0.7163 | Prec: 0.9872

✅ All results consolidated into: Validation_Results_Final/Constitutional_Validation_Level_Sorted.xlsx
